In [1]:
pip install bertopic

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from bertopic import BERTopic
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ai_df = pd.read_csv('AI.csv', encoding='latin1')
ai_df.rename(columns={'Number': 'id'}, inplace=True)
benefits_df = pd.read_csv('dat_benefits_text.csv', encoding='latin1')
risks_df = pd.read_csv('dat_risks_text.csv', encoding='latin1')

In [5]:
ai_df.head()

,id,risks_AI_avg,support_company,gender_bin,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,country,manipulation_check2,weight
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,US,Pass,0.2703
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,US,Pass,0.2709
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,US,Pass,0.6464
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,US,Pass,2.7281
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,US,Fail,0.8643


In [6]:
benefits_df.head()

,id,benefits_text,gender_bin,gender_cat
0,1,Not sure,Man,Man
1,2,No I cannot,Man,Man
2,3,No idea,Woman,Woman
3,4,Efficency,Man,Man
4,5,In medicine,Woman,Woman


In [7]:
risks_df.head()

,id,risks_text,gender_bin,gender_cat
0,2,Not sure,Man,Man
1,3,Taking over the world,Man,Man
2,8,People losing jobs,Woman,Woman
3,10,Job loss,Man,Man
4,12,Crime,Woman,Woman


In [8]:
df = ai_df.merge(benefits_df, on="id", how="left")
df = df.merge(risks_df, on="id", how="left")

In [9]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,country,manipulation_check2,weight,benefits_text,gender_bin_y,gender_cat_x,risks_text,gender_bin,gender_cat_y
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,US,Pass,0.2703,Not sure,Man,Man,NaN,NaN,NaN
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,US,Pass,0.2709,No I cannot,Man,Man,Not sure,Man,Man
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,US,Pass,0.6464,No idea,Woman,Woman,Taking over the world,Man,Man
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,US,Pass,2.7281,Efficency,Man,Man,NaN,NaN,NaN
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,US,Fail,0.8643,In medicine,Woman,Woman,NaN,NaN,NaN


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3049 entries, 0 to 3048
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   3049 non-null   int64  
 1   risks_AI_avg         3049 non-null   float64
 2   support_company      2689 non-null   float64
 3   gender_bin_x         3007 non-null   object 
 4   objective_threat     1600 non-null   float64
 5   trait_risk           3049 non-null   object 
 6   educ_short           3049 non-null   int64  
 7   percent_job_gain     3049 non-null   object 
 8   age_group            3049 non-null   object 
 9   education            3049 non-null   object 
 10  country              3049 non-null   object 
 11  manipulation_check2  3045 non-null   object 
 12  weight               3049 non-null   float64
 13  benefits_text        2891 non-null   object 
 14  gender_bin_y         3001 non-null   object 
 15  gender_cat_x         3001 non-null   o

In [11]:
df.describe()

,id,risks_AI_avg,support_company,objective_threat,educ_short,weight
count,3049.000000,3049.000000,2689.000000,1600.000000,3049.000000,3049.000000
mean,1525.000000,4.653657,3.119747,0.518458,0.299442,0.999236
std,880.314811,2.508140,1.094399,0.218976,0.458089,0.616210
min,1.000000,0.000000,1.000000,0.000000,0.000000,0.137600
25%,763.000000,3.000000,2.000000,0.366667,0.000000,0.647400
50%,1525.000000,5.000000,3.000000,0.500000,0.000000,0.834400
75%,2287.000000,6.000000,4.000000,0.666667,1.000000,1.135600
max,3049.000000,10.000000,5.000000,1.000000,1.000000,5.739000


In [12]:
df["combined_text"] = df[[c for c in df.columns if c.endswith("text")]].fillna("").agg(" ".join, axis=1)

In [13]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,country,manipulation_check2,weight,benefits_text,gender_bin_y,gender_cat_x,risks_text,gender_bin,gender_cat_y,combined_text
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,US,Pass,0.2703,Not sure,Man,Man,NaN,NaN,NaN,Not sure
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,US,Pass,0.2709,No I cannot,Man,Man,Not sure,Man,Man,No I cannot Not sure
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,US,Pass,0.6464,No idea,Woman,Woman,Taking over the world,Man,Man,No idea Taking over the world
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,US,Pass,2.7281,Efficency,Man,Man,NaN,NaN,NaN,Efficency
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,US,Fail,0.8643,In medicine,Woman,Woman,NaN,NaN,NaN,In medicine


# 3. RoBERTa Embeddings (HuggingFace + PyTorch)

In [14]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModel.from_pretrained("roberta-base")
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [15]:
@torch.no_grad()
def get_embeddings(text_list, batch_size=16, max_length=128):
    all_emb = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
        outputs = model(**inputs)
        # mean pooling
        emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        all_emb.append(emb)
    return np.vstack(all_emb)

print("Generating embeddings...")
embeddings = get_embeddings(df["combined_text"].tolist())

Generating embeddings...


# 4. BERTopic (Topic Proportions as Features)

In [16]:
topic_model = BERTopic(
    verbose=True,
    calculate_probabilities=True
)

topics, probs = topic_model.fit_transform(df["combined_text"], embeddings)

df["topic"] = topics
# df["is_outlier"] = (df["topic"] == -1).astype(int)

probs = np.array(probs)

probs_df = pd.DataFrame(
    probs,
    columns=[f"topic_{i}" for i in range(probs.shape[1])]
)

# probs_df = probs_df.div(probs_df.sum(axis=1).replace(0, 1), axis=0)

df = pd.concat([df.reset_index(drop=True), probs_df], axis=1)

# Inspect topics
print(topic_model.get_topic_info().head(10))
for tid in topic_model.get_topic_info()["Topic"].head(8):
    print(f"\nTopic {tid}:")
    print(topic_model.get_topic(tid))

2026-04-13 17:10:02,955 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2026-04-13 17:10:09,906 - BERTopic - Dimensionality - Completed ✓
2026-04-13 17:10:09,907 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 17:10:10,128 - BERTopic - Cluster - Completed ✓
2026-04-13 17:10:10,129 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-13 17:10:10,151 - BERTopic - Representation - Completed ✓


   Topic  Count                                   Name  \
0     -1    994                      -1_and_the_of_for   
1      0    239                    0_to_do_able_enough   
2      1    129                     1_the_of_jobs_away   
3      2     85            2_jobs_losing_taking_people   
4      3     68              3_than_bigger_faster_more   
5      4     67          4_taking_human_jobs_workforce   
6      5     64    5_efficiency_cost_operation_savings   
7      6     63  6_speed_medical_healthcare_convenient   
8      7     59                        7_will_it_is_we   
9      8     59        8_ease_saves_knowledge_increase   

                                      Representation  \
0     [and, the, of, for, be, that, it, not, is, in]   
1  [to, do, able, enough, being, people, make, th...   
2  [the, of, jobs, away, in, for, ai, on, some, f...   
3  [jobs, losing, taking, people, loss, lose, mak...   
4  [than, bigger, faster, more, humans, it, on, t...   
5  [taking, human, jobs, 

# 5. Sentiment (VADER)

In [17]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0.0
    return analyzer.polarity_scores(text)["compound"]  # -1 to 1

print("Computing sentiment...")
df["sentiment"] = df["combined_text"].apply(get_sentiment)

Computing sentiment...


In [18]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,...,topic_55,topic_56,topic_57,topic_58,topic_59,topic_60,topic_61,topic_62,topic_63,sentiment
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,...,0.004425,0.007523,0.002101,0.004945,0.000813,0.003511,0.002553,0.003187,0.002674,-0.2411
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,...,0.046547,0.003891,0.003107,0.019759,0.000586,0.011143,0.003759,0.002857,0.005011,-0.1250
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,...,0.035090,0.004643,0.004305,0.020861,0.000760,0.022061,0.005157,0.003706,0.007483,-0.2960
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,...,0.005636,0.007970,0.003774,0.006113,0.001316,0.006241,0.015167,0.047983,0.006810,0.0000
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,...,0.003064,0.003845,0.002139,0.003298,0.000749,0.003412,0.010073,0.016182,0.003863,0.0000


# 6. Uncertainty Score (0-5)
## Simple lexicon-based scoring

In [19]:
uncertain_terms = [
    r"\bmaybe\b", r"\bnot sure\b", r"\bunsure\b", r"\buncertain\b", r"\bi think\b",
    r"\bprobably\b", r"\bpossibly\b", r"\bcould\b", r"\bmight\b", r"\bperhaps\b",
    r"\bdon'?t know\b", r"\bno idea\b"
]
uncertain_regex = re.compile("|".join(uncertain_terms), flags=re.IGNORECASE)

def uncertainty_score(text, cap=5):
    if not isinstance(text, str) or text.strip() == "":
        return 0
    matches = re.findall(uncertain_regex, text)
    return int(min(len(matches), cap))

print("Computing uncertainty...")
df["uncertainty"] = df["combined_text"].apply(uncertainty_score)

Computing uncertainty...


In [20]:
df.head()

,id,risks_AI_avg,support_company,gender_bin_x,objective_threat,trait_risk,educ_short,percent_job_gain,age_group,education,...,topic_56,topic_57,topic_58,topic_59,topic_60,topic_61,topic_62,topic_63,sentiment,uncertainty
0,1,4.0,4.0,Men,NaN,certain,0,100%,45 to 64,HS or less,...,0.007523,0.002101,0.004945,0.000813,0.003511,0.002553,0.003187,0.002674,-0.2411,1
1,2,4.0,2.0,NaN,0.200000,certain,1,30%,45 to 64,Postgrad,...,0.003891,0.003107,0.019759,0.000586,0.011143,0.003759,0.002857,0.005011,-0.1250,1
2,3,3.0,4.0,Men,NaN,certain,0,70%,45 to 64,HS or less,...,0.004643,0.004305,0.020861,0.000760,0.022061,0.005157,0.003706,0.007483,-0.2960,1
3,4,2.5,3.0,Men,0.466667,certain,0,100%,30 to 44,HS or less,...,0.007970,0.003774,0.006113,0.001316,0.006241,0.015167,0.047983,0.006810,0.0000,0
4,5,3.0,4.0,Men,NaN,certain,0,30%,65 or older,Some college,...,0.003845,0.002139,0.003298,0.000749,0.003412,0.010073,0.016182,0.003863,0.0000,0


In [21]:
print(df.columns.tolist())

['id', 'risks_AI_avg', 'support_company', 'gender_bin_x', 'objective_threat', 'trait_risk', 'educ_short', 'percent_job_gain', 'age_group', 'education', 'country', 'manipulation_check2', 'weight', 'benefits_text', 'gender_bin_y', 'gender_cat_x', 'risks_text', 'gender_bin', 'gender_cat_y', 'combined_text', 'topic', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5', 'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11', 'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17', 'topic_18', 'topic_19', 'topic_20', 'topic_21', 'topic_22', 'topic_23', 'topic_24', 'topic_25', 'topic_26', 'topic_27', 'topic_28', 'topic_29', 'topic_30', 'topic_31', 'topic_32', 'topic_33', 'topic_34', 'topic_35', 'topic_36', 'topic_37', 'topic_38', 'topic_39', 'topic_40', 'topic_41', 'topic_42', 'topic_43', 'topic_44', 'topic_45', 'topic_46', 'topic_47', 'topic_48', 'topic_49', 'topic_50', 'topic_51', 'topic_52', 'topic_53', 'topic_54', 'topic_55', 'topic_56', 'topic_57',

In [22]:
df = df.loc[:, ~df.columns.duplicated()]

In [23]:
df["gender_clean"] = df["gender_bin"]

# Fill missing from other sources
df["gender_clean"] = df["gender_clean"].fillna(df["gender_bin_x"])
df["gender_clean"] = df["gender_clean"].fillna(df["gender_bin_y"])

In [24]:
df["gender_clean"].isna().sum()

np.int64(0)

# 7. Build Feature Set

In [25]:
#structured = [c for c in [
#    "gender", "age", "education", "risk_orientation"
#] if c in df.columns]

structured = [
    "gender_clean",
    "age_group",
    "education",
    "trait_risk",
    "objective_threat",
    "percent_job_gain"
]

topic_cols = [c for c in df.columns if c.startswith("topic_")]
text_feats = topic_cols + ["sentiment", "uncertainty"]

# Targets
risk_target = "risks_AI_avg"            # continuous
support_target = "support_company"     # ordinal (1-5)

# Prepare modeling df
model_cols = structured + text_feats + [risk_target, support_target]
df_m = df[model_cols].dropna().copy()

# Basic encoding (example)
for col in structured:
    if df_m[col].dtype == "object":
        df_m[col] = df_m[col].astype("category").cat.codes

In [26]:
# 8. Model A: Text + Structured -> Risk Perception

In [27]:
X = df_m[structured + text_feats]
y = df_m[risk_target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("\nRisk Model Performance")
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

# Feature importance
feat_imp = pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_})\
    .sort_values("importance", ascending=False)
print("\nTop Features (Risk):")
print(feat_imp.head(20))


Risk Model Performance
MSE: 6.3734883005401235
R2: 0.07062918905044724

Top Features (Risk):
             feature  importance
4   objective_threat    0.082774
70         sentiment    0.054531
1          age_group    0.042438
5   percent_job_gain    0.023461
0       gender_clean    0.022775
2          education    0.022703
6            topic_0    0.022014
12           topic_6    0.021930
3         trait_risk    0.019224
49          topic_43    0.018823
16          topic_10    0.018253
26          topic_20    0.016387
13           topic_7    0.016244
23          topic_17    0.015355
7            topic_1    0.014772
36          topic_30    0.014305
14           topic_8    0.014182
58          topic_52    0.013642
67          topic_61    0.013302
64          topic_58    0.013270


In [28]:
# 9. Model B: Risk -> AI Support (Ordinal via OLS approx)
# (For proper ordinal, use statsmodels OrderedModel)

In [29]:
X2 = df_m[[risk_target] + structured + text_feats]
y2 = df_m[support_target]

X2 = sm.add_constant(X2)
ols_support = sm.OLS(y2, X2).fit(cov_type='HC3')
print("\nSupport Model (OLS approx):")
print(ols_support.summary())


Support Model (OLS approx):
                            OLS Regression Results                            
Dep. Variable:        support_company   R-squared:                       0.146
Model:                            OLS   Adj. R-squared:                  0.101
Method:                 Least Squares   F-statistic:                     3.562
Date:                Mon, 13 Apr 2026   Prob (F-statistic):           2.69e-20
Time:                        17:10:17   Log-Likelihood:                -2036.4
No. Observations:                1436   AIC:                             4221.
Df Residuals:                    1362   BIC:                             4611.
Df Model:                          73                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const      

In [30]:
# 10. Mediation Analysis (Baron & Kenny style)
# Path: Text -> Risk -> Support

In [31]:
# Step 1: M ~ X (Risk on Text+Structured)
formula_m = f"{risk_target} ~ " + " + ".join(structured + text_feats)
model_m = smf.ols(formula_m, data=df_m).fit(cov_type='HC3')

# Step 2: Y ~ X (Support on Text+Structured)
formula_yx = f"{support_target} ~ " + " + ".join(structured + text_feats)
model_yx = smf.ols(formula_yx, data=df_m).fit(cov_type='HC3')

# Step 3: Y ~ M + X (Support on Risk + Text+Structured)
formula_y = f"{support_target} ~ {risk_target} + " + " + ".join(structured + text_feats)
model_y = smf.ols(formula_y, data=df_m).fit(cov_type='HC3')

print("\nMediation Results:")
print("\nModel M (Risk ~ X):\n", model_m.summary())
print("\nModel Y|X (Support ~ X):\n", model_yx.summary())
print("\nModel Y (Support ~ Risk + X):\n", model_y.summary())


Mediation Results:

Model M (Risk ~ X):
                             OLS Regression Results                            
Dep. Variable:           risks_AI_avg   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     3.888
Date:                Mon, 13 Apr 2026   Prob (F-statistic):           3.10e-23
Time:                        17:10:17   Log-Likelihood:                -3292.4
No. Observations:                1436   AIC:                             6731.
Df Residuals:                    1363   BIC:                             7115.
Df Model:                          72                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

In [32]:
def bootstrap_indirect_effect(df, x_cols, m_col, y_col, n_boot=500):
    inds = []
    for _ in range(n_boot):
        samp = df.sample(frac=1.0, replace=True)
        m = smf.ols(f"{m_col} ~ " + " + ".join(x_cols), data=samp).fit()
        y = smf.ols(f"{y_col} ~ {m_col} + " + " + ".join(x_cols), data=samp).fit()
        # aggregate indirect effect as mean over Xs: sum(a_i * b)
        b = y.params.get(m_col, 0)
        a = np.array([m.params.get(col, 0) for col in x_cols])
        inds.append((a * b).sum())
    return np.percentile(inds, [2.5, 50, 97.5])

ci = bootstrap_indirect_effect(df_m, structured + text_feats, risk_target, support_target)
print("\nBootstrap indirect effect CI (2.5%, 50%, 97.5%):", ci)


Bootstrap indirect effect CI (2.5%, 50%, 97.5%): [-7.80516011  2.19776517 10.50688379]


In [33]:
# 11. Research Question Checks
# Do language differences explain gender gaps?

In [34]:
# Compare gender coefficient with/without text features in Risk model
#if "gender" in df_m.columns:
#    base_cols = [c for c in structured if c != "gender"]
#    m_base = smf.ols(f"{risk_target} ~ gender + " + " + ".join(base_cols), data=df_m).fit(cov_type='HC3')
#    m_text = smf.ols(f"{risk_target} ~ gender + " + " + ".join(base_cols + text_feats), data=df_m).fit(cov_type='HC3')
#    print("\nGender effect WITHOUT text features:", m_base.params.get("gender"))
#    print("Gender effect WITH text features:", m_text.params.get("gender"))

In [36]:
# ── Clean attenuation test: same sample, same controls, only text features differ ──

# Use only fully-observed structured variables (no objective_threat)
base_structured = ["age_group", "education", "trait_risk", "percent_job_gain"]

# Build analysis sample: drop rows missing on these variables + outcome + gender
test_cols = ["gender_clean", risk_target] + base_structured + text_feats
df_test = df[test_cols].dropna()
print(f"Analysis sample N = {len(df_test):,}")

# Model 1: gender + demographics only (NO text features)
formula_base = f"{risk_target} ~ gender_clean + " + " + ".join(base_structured)
m_base = smf.ols(formula_base, data=df_test).fit(cov_type='HC3')
#coef_base = m_base.params.get("gender_clean")
#p_base    = m_base.pvalues.get("gender_clean")
coef_base = m_base.params.get("gender_clean[T.Women]")
p_base    = m_base.pvalues.get("gender_clean[T.Women]")

# Model 2: gender + demographics + text features
formula_text = f"{risk_target} ~ gender_clean + " + " + ".join(base_structured + text_feats)
m_text = smf.ols(formula_text, data=df_test).fit(cov_type='HC3')
#coef_text = m_text.params.get("gender_clean")
#p_text    = m_text.pvalues.get("gender_clean")
coef_text = m_text.params.get("gender_clean[T.Women]")
p_text    = m_text.pvalues.get("gender_clean[T.Women]")

attenuation = (coef_base - coef_text) / coef_base * 100

print(f"\nGender effect WITHOUT text features: {coef_base:+.4f}  (p={p_base:.4f})")
print(f"Gender effect WITH text features:    {coef_text:+.4f}  (p={p_text:.4f})")
print(f"Attenuation: {attenuation:.1f}%")
print(f"R² without text: {m_base.rsquared:.4f}")
print(f"R² with text:    {m_text.rsquared:.4f}")
print(f"\nInterpretation: language features explain {attenuation:.0f}% of the gender gap")
print(f"in risk perception on a consistent N={len(df_test):,} sample.")

Analysis sample N = 3,049

Gender effect WITHOUT text features: +0.2984  (p=0.0189)
Gender effect WITH text features:    +0.1811  (p=0.2143)
Attenuation: 39.3%
R² without text: 0.0516
R² with text:    0.0758

Interpretation: language features explain 39% of the gender gap
in risk perception on a consistent N=3,049 sample.
